# Stage 9A — Release decision and full-coverage plan

**Goal:** Turn the monitoring findings into a saved release decision and a concrete plan to collect every day in the next training and evaluation windows.

**Why:** MLOps includes deciding whether a model is safe to use. The current API is a classroom demo. We will hold this model from operational release because important late requests were missed and the one-day-per-month sample has poor winter coverage.

Stage 9B will build the resumable daily collector. This stage records exactly what that collector must gather and which later dates stay untouched for the next final test.

In [ ]:
# Cell 2 — Why: load the exact monitoring evidence and verify it.
from google.colab import drive
from pathlib import Path
import hashlib, json

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/nyc311_mlops")
report_id = "20260926T125109767152Z"
report_dir = root / "monitoring" / report_id
report_file = report_dir / "report.json"
raw_report = report_file.read_bytes()
report = json.loads(raw_report)
if report["report_id"] != report_id or report["kind"] != "offline historical simulation":
    raise ValueError("Wrong monitoring report")
for name, key in [("offline_predictions.jsonl", "prediction_sha256"),
                  ("matured_outcomes.jsonl", "outcome_sha256")]:
    if hashlib.sha256((report_dir / name).read_bytes()).hexdigest() != report[key]:
        raise ValueError(f"Changed monitoring file: {name}")
print("Verified report:", report_id, "Rows:", report["rows_joined"])

In [ ]:
# Cell 3 — Why: connect the report to the exact saved model.
run_id = "20260926T120702352046Z"
run_dir = root / "runs" / run_id
run = json.loads((run_dir / "run.json").read_text())
model_file = run_dir / "model.joblib"
if run["run_id"] != run_id or report["model_run_id"] != run_id:
    raise ValueError("Report and model versions differ")
if hashlib.sha256(model_file.read_bytes()).hexdigest() != run["model_sha256"]:
    raise ValueError("Saved model changed")
print("Verified model:", run_id)
print("Overall test recall:", report["test_overall"]["recall"])
print("December recall:", report["monthly"]["12"]["recall"])

In [ ]:
# Cell 4 — Why: state the observed reasons for review clearly.
snow = report["by_type_min_300"]["Snow or Ice"]
heat = report["by_type_min_300"]["HEAT/HOT WATER"]
brooklyn = report["by_borough_min_100"]["BROOKLYN"]
reasons = []
if report["review_needed"]:
    reasons.append(f"Validation-to-test recall gap: {report['recall_gap']}")
if snow["late_rate"] > 0 and snow["recall"] == 0:
    reasons.append("Snow or Ice: late cases present, recall zero")
if heat["late_rate"] > 0 and heat["recall"] == 0:
    reasons.append("HEAT/HOT WATER: late cases present, recall zero")
if brooklyn["recall"] < report["test_overall"]["recall"]:
    reasons.append(f"Brooklyn recall below overall: {brooklyn['recall']}")
print("Reasons to hold:")
for reason in reasons:
    print("-", reason)

In [ ]:
# Cell 5 — Why: define complete daily coverage and a fresh untouched test.
from datetime import date

windows = {
    "train": ["2025-01-01", "2026-01-01"],
    "validation": ["2026-01-01", "2026-03-01"],
    "test": ["2026-03-01", "2026-05-01"]
}
days_planned = (date.fromisoformat(windows["test"][1]) -
                date.fromisoformat(windows["train"][0])).days
if days_planned != 485:
    raise ValueError("Unexpected date coverage")
print("Daily partitions planned:", days_planned)
print("Train / validation / untouched test:", windows)

In [ ]:
# Cell 6 — Why: save the release decision with its evidence.
from datetime import datetime, timezone

decision_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
decision_dir = root / "release_decisions" / decision_id
decision_dir.mkdir(parents=True, exist_ok=False)
decision = {
    "decision_id": decision_id, "model_run_id": run_id,
    "monitoring_report_id": report_id,
    "monitoring_report_sha256": hashlib.sha256(raw_report).hexdigest(),
    "decision": "HOLD_FOR_OPERATIONAL_USE",
    "classroom_api_status": "demo_only", "reasons": reasons,
    "next_step": "Collect complete daily coverage before a new candidate",
    "lesson_version": "stage9a-v1"
}
(decision_dir / "decision.json").write_text(json.dumps(decision, indent=2) + "\n", encoding="utf-8")
print("Decision ID:", decision_id)
print("Release decision:", decision["decision"])

In [ ]:
# Cell 7 — Why: save one collection plan for the next pipeline.
plan = {
    "plan_id": decision_id, "source_dataset": "erm2-nwe9",
    "source_url": "https://data.cityofnewyork.us/resource/erm2-nwe9.json",
    "fields": ["unique_key", "created_date", "closed_date", "complaint_type", "borough"],
    "coverage": "every calendar day, all requests, no one-day-per-month sampling",
    "windows_start_inclusive_end_exclusive": windows,
    "daily_partitions_planned": days_planned,
    "test_sealed_for_model_selection": True,
    "storage_rule": "one verified raw file and SHA-256 manifest per day",
    "resume_rule": "verify completed days and continue with missing days",
    "source_release_decision_id": decision_id
}
(decision_dir / "coverage_plan.json").write_text(json.dumps(plan, indent=2) + "\n", encoding="utf-8")
print("Saved decision and coverage plan:", decision_dir)

# Stage 9A takeaway and quiz

**What you learned:** A model can be technically reproducible and have a working API while still being held from operational use. We saved the reasons and a plan for all 485 calendar days. The new March–April 2026 test window stays untouched while we build and choose the next model.

**Easy quiz:**

1. Why is the current API marked `demo_only`?
2. Why does the collection plan require every day instead of one day per month?
3. Which dates are the next validation window? Which dates are the sealed test window?
4. Why must we avoid using that new test window to choose a model or threshold?

Send me the **Decision ID**, **Release decision**, **Daily partitions planned**, saved path, and your answers. Stage 9B will build the resumable collector.